In [2]:
# =============================================================================
# Spatial Confusion Map — Two Specific Days Side by Side
# July 24, 2022  |  July 19, 2021
# XGBoost ignition model (model_ignition_final.pkl)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
import glob
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('figures', exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR = r'C:\Users\lambe\RU_Thesis_2026\ml_data'

DYNAMIC_FEATURES = [
    'temp_max', 'rh_min', 'vpd_mean', 'wind_speed_mean',
    'precip_sum', 'solar_rad_mj', 'rh_7d', 'temp_7d',
    'precip_30d', 'recovery_value'
]
STATIC_FEATURES = [
    'elevation_m', 'slope_deg', 'land_cover',
    'pop_density', 'road_proximity_m'
]
FEATURES       = DYNAMIC_FEATURES + STATIC_FEATURES
VAL_YEARS      = [2021, 2022]
DOY_START_HARD = 91
DOY_END_HARD   = 310

# Target days — format YYYYMMDD
TARGET_DAYS = ['20220724', '20210719']

COLORS = {
    'TN': '#a8d8ea',   # light blue
    'TP': '#e74c3c',   # bright red
    'FN': '#0c0c0c',   # black
    'FP': '#f39c12',   # orange
}
PLOT_ORDER = ['TN', 'FP', 'TP', 'FN']
ALPHAS     = {'TN': 0.75, 'FP': 0.90, 'TP': 0.95, 'FN': 1.00}
ZORDERS    = {'TN': 2,    'FP': 3,    'TP': 4,    'FN': 5}
PIXEL_STEP = 0.09   # ~9 km in degrees

# ── Load model ─────────────────────────────────────────────────────────────────
print("Loading final ignition model...")
bundle       = joblib.load('models/model_ignition_final.pkl')
model_ign    = bundle['model']
CV_THRESHOLD = bundle['cv_threshold']
print(f"  CV threshold: {CV_THRESHOLD:.4f}")

# ── Load validation data ───────────────────────────────────────────────────────
print("\nLoading validation data...")

KEEP_COLS = ['date', 'longitude', 'latitude', 'target_ignition'] + FEATURES

val_dfs = []
for year in VAL_YEARS:
    for f in sorted(glob.glob(
            os.path.join(DATA_DIR, f'ML_val_{year}??.csv'))):
        try:
            df = pd.read_csv(f, usecols=KEEP_COLS, on_bad_lines='skip')
            val_dfs.append(df)
        except Exception as e:
            print(f"  WARNING: {os.path.basename(f)}: {e}")

df_val = pd.concat(val_dfs, ignore_index=True)
df_val['date'] = df_val['date'].astype(int).astype(str)
df_val['doy']  = pd.to_datetime(
    df_val['date'], format='%Y%m%d', errors='coerce').dt.dayofyear
df_val = df_val[(df_val['doy'] >= DOY_START_HARD) &
                (df_val['doy'] <= DOY_END_HARD)].drop(columns='doy')
df_val = df_val.dropna(
    subset=FEATURES + ['target_ignition', 'longitude', 'latitude'])
df_val['target_ignition'] = df_val['target_ignition'].astype(int)

print(f"  Validation rows: {len(df_val):,}")

# ── Run predictions on full validation set ─────────────────────────────────────
print("\nRunning predictions...")

X_val        = df_val[FEATURES].values
y_true       = df_val['target_ignition'].values
y_prob       = model_ign.predict_proba(X_val)[:, 1]
y_pred       = (y_prob >= CV_THRESHOLD).astype(int)

df_val['y_true'] = y_true
df_val['y_pred'] = y_pred

def assign_conf(row):
    if   row['y_true'] == 0 and row['y_pred'] == 0: return 'TN'
    elif row['y_true'] == 1 and row['y_pred'] == 1: return 'TP'
    elif row['y_true'] == 1 and row['y_pred'] == 0: return 'FN'
    else:                                             return 'FP'

df_val['conf_class'] = df_val.apply(assign_conf, axis=1)

# ── Verify target days exist ───────────────────────────────────────────────────
print(f"\nVerifying target days...")
for day in TARGET_DAYS:
    n = (df_val['date'] == day).sum()
    if n == 0:
        raise ValueError(
            f"Day {day} not found in validation data. "
            f"Available dates range: "
            f"{df_val['date'].min()} – {df_val['date'].max()}")
    n_fire = ((df_val['date'] == day) &
              (df_val['target_ignition'] == 1)).sum()
    n_tp   = ((df_val['date'] == day) &
              (df_val['conf_class'] == 'TP')).sum()
    n_fp   = ((df_val['date'] == day) &
              (df_val['conf_class'] == 'FP')).sum()
    n_fn   = ((df_val['date'] == day) &
              (df_val['conf_class'] == 'FN')).sum()
    n_tn   = ((df_val['date'] == day) &
              (df_val['conf_class'] == 'TN')).sum()
    print(f"  {day}: {n:,} pixels  |  "
          f"fires={n_fire}  TP={n_tp}  FP={n_fp}  "
          f"FN={n_fn}  TN={n_tn:,}")

# ── Prepare per-day dataframes ─────────────────────────────────────────────────
day_data = {}
for day in TARGET_DAYS:
    df_d = df_val[df_val['date'] == day].copy()
    df_d['lon_r'] = df_d['longitude'].round(4)
    df_d['lat_r'] = df_d['latitude'].round(4)
    df_d = df_d.drop_duplicates(subset=['lon_r', 'lat_r'], keep='last')

    n_tp = (df_d['conf_class'] == 'TP').sum()
    n_fp = (df_d['conf_class'] == 'FP').sum()
    n_fn = (df_d['conf_class'] == 'FN').sum()
    n_tn = (df_d['conf_class'] == 'TN').sum()

    recall = n_tp / max(n_tp + n_fn, 1)
    fpr    = n_fp / max(n_fp + n_tn, 1)
    prec   = n_tp / max(n_tp + n_fp, 1)

    day_data[day] = {
        'df'    : df_d,
        'n_tp'  : n_tp,
        'n_fp'  : n_fp,
        'n_fn'  : n_fn,
        'n_tn'  : n_tn,
        'n_fire': n_tp + n_fn,
        'recall': recall,
        'fpr'   : fpr,
        'prec'  : prec,
    }

# ── Compute shared dot size from full domain extent ────────────────────────────
# Use the union of both days for consistent sizing across panels
all_lons = pd.concat([d['df']['lon_r'] for d in day_data.values()])
all_lats = pd.concat([d['df']['lat_r'] for d in day_data.values()])
LON_RANGE = all_lons.max() - all_lons.min()
LAT_RANGE = all_lats.max() - all_lats.min()

FIG_W, FIG_H   = 20, 10
DPI            = 300
AXES_W_EACH    = (FIG_W / 2) * 0.78
AXES_H         = FIG_H * 0.78

PX_PER_DEG_X  = (AXES_W_EACH * DPI) / LON_RANGE
PX_PER_DEG_Y  = (AXES_H      * DPI) / LAT_RANGE
PIXEL_PX      = PIXEL_STEP * min(PX_PER_DEG_X, PX_PER_DEG_Y)
POINTS_PER_PX = DPI / 72
S_BASE        = (PIXEL_PX / POINTS_PER_PX) ** 2 * 0.72

S_SIZES = {
    'TN': S_BASE * 0.85,
    'FP': S_BASE * 1.1,
    'TP': S_BASE * 1.3,
    'FN': S_BASE * 1.6,
}

print(f"\n  Dot size (s): {S_BASE:.1f}")

# ── Shared axis limits (same domain for both panels) ──────────────────────────
LON_MIN = float(all_lons.min()) - 0.5
LON_MAX = float(all_lons.max()) + 0.5
LAT_MIN = float(all_lats.min()) - 0.3
LAT_MAX = float(all_lats.max()) + 0.3

# ── Figure ─────────────────────────────────────────────────────────────────────
print(f"\nGenerating two-panel confusion map...")

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))
fig.patch.set_facecolor('white')
fig.subplots_adjust(wspace=0.08)

for ax, day in zip(axes, TARGET_DAYS):
    info     = day_data[day]
    df_d     = info['df']
    date_obj = datetime.strptime(day, '%Y%m%d')
    date_str = date_obj.strftime('%B %d, %Y')
    date_doy = date_obj.timetuple().tm_yday

    # Plot confusion classes
    for cls in PLOT_ORDER:
        subset = df_d[df_d['conf_class'] == cls]
        if len(subset) == 0:
            continue
        ax.scatter(
            subset['lon_r'], subset['lat_r'],
            c          = COLORS[cls],
            s          = S_SIZES[cls],
            alpha      = ALPHAS[cls],
            linewidths = 0,
            rasterized = True,
            zorder     = ZORDERS[cls],
            marker     = 's'
        )

    # Shared axis limits — same domain both panels
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)

    # Labels
    ax.set_xlabel('Longitude (°W)', fontsize=10)
    ax.set_ylabel('Latitude (°N)',  fontsize=10)
    ax.set_title(
        f'{date_str}  (DOY {date_doy})\n'
        f'TP = {info["n_tp"]}  '
        f'FN = {info["n_fn"]}  '
        f'FP = {info["n_fp"]:,}  '
        f'TN = {info["n_tn"]:,}',
        fontsize=11, fontweight='bold', pad=10
    )
    ax.tick_params(labelsize=9)
    ax.grid(color='#e8e8e8', linewidth=0.4, alpha=0.7, zorder=1)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    # Per-panel stats box
    stats_text = (
        f"Actual fires : {info['n_fire']}\n"
        f"Recall       : {info['recall']:.3f}\n"
        f"Precision    : {info['prec']:.4f}\n"
        f"FPR          : {info['fpr']:.3f}"
    )
    ax.text(
        0.985, 0.985, stats_text,
        transform           = ax.transAxes,
        fontsize            = 8.5,
        verticalalignment   = 'top',
        horizontalalignment = 'right',
        fontfamily          = 'monospace',
        bbox                = dict(boxstyle='round,pad=0.5',
                                   facecolor='white',
                                   edgecolor='#cccccc',
                                   alpha=0.95)
    )

# ── Shared legend (below both panels) ─────────────────────────────────────────
legend_labels = {
    'TN': 'True Negative  — No fire, correctly predicted',
    'TP': 'True Positive  — Fire correctly detected',
    'FP': 'False Positive — No fire, incorrectly flagged',
    'FN': 'False Negative — Fire missed',
}
legend_patches = [
    mpatches.Patch(
        facecolor = COLORS[cls],
        edgecolor = '#888888' if cls == 'TN' else 'none',
        linewidth = 0.5,
        label     = legend_labels[cls])
    for cls in ['TN', 'TP', 'FP', 'FN']
]
fig.legend(
    handles        = legend_patches,
    fontsize       = 9,
    loc            = 'lower center',
    ncol           = 4,
    framealpha     = 0.95,
    edgecolor      = '#cccccc',
    bbox_to_anchor = (0.5, -0.04),
    title          = 'Prediction outcome',
    title_fontsize = 9,
)

# ── Overall title ──────────────────────────────────────────────────────────────
fig.suptitle(
    'Spatial Confusion Maps — Ontario Wildfire Ignition Model\n'
    f'XGBoost (Optuna)  |  Threshold = {CV_THRESHOLD:.3f}',
    fontsize=13, fontweight='bold', y=1.02
)
fig.text(
    0.5, -0.08,
    'Ontario wildfire ignition model  |  XGBoost (Optuna)  |  '
    'Source: Canadian National Fire Database (CNFDB)',
    ha='center', fontsize=8, color='#888888', style='italic'
)

plt.tight_layout()

fig.savefig('figures/fig_confusion_map_two_days.png',
            dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('figures/fig_confusion_map_two_days.pdf',
            bbox_inches='tight', facecolor='white')

print("  Saved: figures/fig_confusion_map_two_days.png")
print("  Saved: figures/fig_confusion_map_two_days.pdf")
plt.close(fig)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("SUMMARY")
print(f"{'='*55}")
print(f"  Model     : model_ignition_final.pkl")
print(f"  Threshold : {CV_THRESHOLD:.4f}")
print(f"\n  {'Day':<14} {'Fires':>6} {'TP':>5} {'FN':>5} "
      f"{'FP':>8} {'Recall':>8} {'FPR':>8}")
print(f"  {'─'*58}")
for day, info in day_data.items():
    date_str = datetime.strptime(day, '%Y%m%d').strftime('%b %d, %Y')
    print(f"  {date_str:<14} {info['n_fire']:>6} {info['n_tp']:>5} "
          f"{info['n_fn']:>5} {info['n_fp']:>8,} "
          f"{info['recall']:>8.3f} {info['fpr']:>8.3f}")

print(f"\nLaTeX:")
print(f"  \\includegraphics[width=\\textwidth]"
      f"{{figures/fig_confusion_map_two_days}}")

Loading final ignition model...
  CV threshold: 0.7290

Loading validation data...
  Validation rows: 5,221,040

Running predictions...

Verifying target days...
  20220724: 11,866 pixels  |  fires=5  TP=5  FP=1162  FN=0  TN=10,699
  20210719: 11,866 pixels  |  fires=98  TP=95  FP=4053  FN=3  TN=7,715

  Dot size (s): 4.3

Generating two-panel confusion map...
  Saved: figures/fig_confusion_map_two_days.png
  Saved: figures/fig_confusion_map_two_days.pdf

SUMMARY
  Model     : model_ignition_final.pkl
  Threshold : 0.7290

  Day             Fires    TP    FN       FP   Recall      FPR
  ──────────────────────────────────────────────────────────
  Jul 24, 2022        5     5     0    1,162    1.000    0.098
  Jul 19, 2021       98    95     3    4,053    0.969    0.344

LaTeX:
  \includegraphics[width=\textwidth]{figures/fig_confusion_map_two_days}
